# 02 — L-BFGS Experiments

Esperimenti con l'algoritmo L-BFGS (A1) per il problema:
$$\min_w \|\hat{X}w - \hat{y}\|$$

Confronto tra:
- Exact line search vs Strong Wolfe line search
- Tolleranza relativa vs assoluta
- Effetto del parametro di memoria `m_history`
- Effetto del parametro di regolarizzazione $\lambda$

In [3]:
import sys
sys.path.insert(0, '../..')

import numpy as np
import matplotlib.pyplot as plt
# Imposta un font che supporta i simboli matematici unicode
plt.rcParams['font.family'] = 'DejaVu Sans'

from utils import (compute_loss, solve_exact, relative_error,
                       residual_norm, compute_condition_number)
from lbfgs import lbfgs_optimize

## Setup dati

In [4]:
np.random.seed(42)
m, n = 600, 12
lam = 0.5

X = np.random.randn(m, n)
y = np.random.randn(n)

# Soluzione esatta (baseline)
w_star, f_star = solve_exact(X, y, lam)
kappa = compute_condition_number(X, lam)
print(f"m={m}, n={n}, λ={lam}")
print(f"κ(X̂) = {kappa:.2f}")
print(f"f(w*) = {f_star:.10e}")

ValueError: too many values to unpack (expected 2)

## 2.1 Exact LS vs Wolfe LS

In [ ]:
# Exact line search
w_ex, h_ex, t_ex = lbfgs_optimize(
    X, y, lam, m_history=10, tol=1e-14,
    tol_type='relative', line_search='exact', verbose=True)

print(f"\n||w-w*|| = {np.linalg.norm(w_ex - w_star):.2e}")
print(f"Iterazioni: {len(h_ex['f'])-1}, Tempo: {t_ex:.4f}s")

In [ ]:
# Wolfe line search
w_wo, h_wo, t_wo = lbfgs_optimize(
    X, y, lam, m_history=10, tol=1e-14,
    tol_type='relative', line_search='wolfe', verbose=True)

print(f"\n||w-w*|| = {np.linalg.norm(w_wo - w_star):.2e}")
print(f"Iterazioni: {len(h_wo['f'])-1}, Tempo: {t_wo:.4f}s")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].semilogy(h_ex['f'], lw=2, label='Exact LS')
axes[0].semilogy(h_wo['f'], lw=2, ls='--', label='Wolfe LS')
axes[0].axhline(f_star, color='r', ls=':', label='f(w*)')
axes[0].set(xlabel='Iterazione', ylabel='f(w)', title='Convergenza f(w)')
axes[0].legend(); axes[0].grid(alpha=0.3)

axes[1].semilogy(h_ex['grad_norm'], lw=2, label='Exact LS')
axes[1].semilogy(h_wo['grad_norm'], lw=2, ls='--', label='Wolfe LS')
axes[1].set(xlabel='Iterazione', ylabel=r"$||\nabla f||$", title=r"$||\nabla f(w)||$")
axes[1].legend(); axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.show()

## 2.2 Effetto del parametro di memoria `m_history`

In [ ]:
results_mem = {}
for mh in [3, 5, 10, 20, 40]:
    w_m, h_m, t_m = lbfgs_optimize(
        X, y, lam, m_history=mh, tol=1e-12,
        tol_type='relative', line_search='exact', verbose=False)
    results_mem[mh] = h_m
    print(f"  m_history={mh:3d}  iter={len(h_m['f'])-1:4d}  "
          f"||w-w*||={np.linalg.norm(w_m-w_star):.2e}  t={t_m:.4f}s")

In [ ]:
plt.figure(figsize=(8, 5))
for mh, h in results_mem.items():
    plt.semilogy(h['grad_norm'], label=f'm={mh}')
plt.xlabel('Iterazione'); plt.ylabel('||∇f||')
plt.title('Effetto del parametro di memoria m_history')
plt.legend(); plt.grid(alpha=0.3)
plt.show()

## 2.3 Effetto di $\lambda$ (condizionamento)

In [ ]:
results_lam = {}
for lam_val in [1e-4, 1e-2, 1.0, 1e2, 1e4]:
    w_l, h_l, t_l = lbfgs_optimize(
        X, y, lam_val, m_history=10, tol=1e-10,
        tol_type='relative', line_search='exact', verbose=False,
        max_iter=50000)
    w_star_l, f_star_l = solve_exact(X, y, lam_val)
    kappa_l = compute_condition_number(X, lam_val)
    results_lam[lam_val] = h_l
    print(f"  λ={lam_val:<8}  κ={kappa_l:>12.1f}  "
          f"iter={len(h_l['f'])-1:5d}  "
          f"||w-w*||={np.linalg.norm(w_l-w_star_l):.2e}  t={t_l:.3f}s")

In [ ]:
plt.figure(figsize=(8, 5))
for lam_val, h in results_lam.items():
    plt.semilogy(h['grad_norm'], label=f'λ={lam_val}')
plt.xlabel('Iterazione'); plt.ylabel('||∇f||')
plt.title('Effetto di λ sulla convergenza L-BFGS')
plt.legend(); plt.grid(alpha=0.3)
plt.show()